# Medical NER Extractor Comparison

This notebook compares different medical entity extractors on the same test sentences.

**Goal**: Find which extractor gives cleaner results (Drug, Disease, Symptom) with less noise.

## Extractors we'll test:
1. **MedCAT** (current) - via Modal API
2. **scispaCy** - with UMLS linking
3. **BioBERT** - HuggingFace model
4. **GLiNER** - Zero-shot NER


In [1]:
# Run this cell to install required packages
!pip install -q scispacy transformers gliner
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


  Preparing metadata (setup.py) ... done


In [2]:
# Test sentences - same for all extractors
TEST_SENTENCES = [
    "Metformin is used to treat Type 2 Diabetes",
    "Patient takes Aspirin and Lisinopril daily for hypertension",
    "65 year old male with history of hypertension and diabetes mellitus on Metformin 500mg",
    "Ibuprofen may cause gastrointestinal bleeding",
    "The weather is nice today",  # Should return nothing
]

print("Test sentences loaded:")
for i, s in enumerate(TEST_SENTENCES, 1):
    print(f"{i}. {s}")


Test sentences loaded:
1. Metformin is used to treat Type 2 Diabetes
2. Patient takes Aspirin and Lisinopril daily for hypertension
3. 65 year old male with history of hypertension and diabetes mellitus on Metformin 500mg
4. Ibuprofen may cause gastrointestinal bleeding
5. The weather is nice today


In [5]:
import requests

MEDCAT_URL = "https://asmaamhadir--medcat-api-fastapi-app.modal.run"

def extract_medcat(text):
    """Extract entities using MedCAT Modal API"""
    try:
        response = requests.post(
            f"{MEDCAT_URL}/extract",
            json={"text": text},
            timeout=60
        )
        data = response.json()
        return [
            {
                "text": e["text"],
                "type": e.get("type", "Medical"),
                "name": e.get("name", e["text"])
            }
            for e in data.get("entities", [])
        ]
    except Exception as e:
        return [{"error": str(e)}]

# Test MedCAT
print("=== MedCAT Results ===\n")
for sentence in TEST_SENTENCES:
    print(f"Input: {sentence}")
    entities = extract_medcat(sentence)
    for e in entities:
        print(f"  → {e})")
    print()


=== MedCAT Results ===

Input: Metformin is used to treat Type 2 Diabetes
  → {'text': 'Metformin', 'type': 'Medical', 'name': 'Metformin'})
  → {'text': 'Type 2 Diabetes', 'type': 'Medical', 'name': 'Diabetes mellitus type 2'})

Input: Patient takes Aspirin and Lisinopril daily for hypertension
  → {'text': 'Patient', 'type': 'Medical', 'name': 'Patient'})
  → {'text': 'takes', 'type': 'Medical', 'name': 'Take - dosing instruction imperative'})
  → {'text': 'Aspirin', 'type': 'Medical', 'name': 'Aspirin'})
  → {'text': 'Lisinopril', 'type': 'Medical', 'name': 'Lisinopril'})
  → {'text': 'daily', 'type': 'Medical', 'name': 'Daily'})
  → {'text': 'hypertension', 'type': 'Medical', 'name': 'Hypertensive disorder, systemic arterial'})

Input: 65 year old male with history of hypertension and diabetes mellitus on Metformin 500mg
  → {'text': 'year', 'type': 'Medical', 'name': 'per year'})
  → {'text': 'male', 'type': 'Medical', 'name': 'Male structure'})
  → {'text': 'history of hypertensi

In [6]:
import spacy

# Load scispaCy model
nlp_sci = spacy.load("en_core_sci_sm")

def extract_scispacy(text):
    """Extract entities using scispaCy"""
    doc = nlp_sci(text)
    return [
        {
            "text": ent.text,
            "type": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        }
        for ent in doc.ents
    ]

# Test scispaCy
print("=== scispaCy Results ===\n")
for sentence in TEST_SENTENCES:
    print(f"Input: {sentence}")
    entities = extract_scispacy(sentence)
    if entities:
        for e in entities:
            print(f"  → {e['text']} ({e['type']})")
    else:
        print("  → (no entities)")
    print()


/home/asmaa/greatness/medclaimsverenv/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


=== scispaCy Results ===

Input: Metformin is used to treat Type 2 Diabetes
  → Metformin (ENTITY)
  → treat (ENTITY)
  → Type 2 Diabetes (ENTITY)

Input: Patient takes Aspirin and Lisinopril daily for hypertension
  → Patient (ENTITY)
  → Aspirin (ENTITY)
  → Lisinopril daily (ENTITY)
  → hypertension (ENTITY)

Input: 65 year old male with history of hypertension and diabetes mellitus on Metformin 500mg
  → male (ENTITY)
  → history (ENTITY)
  → hypertension (ENTITY)
  → diabetes mellitus (ENTITY)
  → Metformin (ENTITY)

Input: Ibuprofen may cause gastrointestinal bleeding
  → Ibuprofen (ENTITY)
  → gastrointestinal bleeding (ENTITY)

Input: The weather is nice today
  → weather (ENTITY)
  → nice (ENTITY)



In [ ]:
from transformers import pipeline

# Load BioBERT NER model for diseases
ner_biobert = pipeline(
    "ner", 
    model="ugaray96/biobert_ncbi_disease_ner",
    aggregation_strategy="simple"
)

def extract_biobert(text):
    """Extract entities using BioBERT"""
    results = ner_biobert(text)
    return [
        {
            "text": r["word"],
            "type": r["entity_group"],
            "score": round(r["score"], 3)
        }
        for r in results
    ]

# Test BioBERT
print("=== BioBERT Results ===\n")
for sentence in TEST_SENTENCES:
    print(f"Input: {sentence}")
    entities = extract_biobert(sentence)
    if entities:
        for e in entities:
            print(f"  → {e['text']} ({e['type']}, score={e['score']})")
    else:
        print("  → (no entities)")
    print()


/home/asmaa/greatness/medclaimsverenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
!pip install numpy==1.26.4


In [3]:
from gliner import GLiNER

# Load GLiNER model
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

# Define the entity types we want
ENTITY_TYPES = ["drug", "disease", "symptom"]

def extract_gliner(text):
    """Extract entities using GLiNER with custom entity types"""
    entities = gliner_model.predict_entities(text, ENTITY_TYPES)
    return [
        {
            "text": e["text"],
            "type": e["label"].upper(),
            "score": round(e["score"], 3)
        }
        for e in entities
    ]

# Test GLiNER
print("=== GLiNER Results ===\n")
for sentence in TEST_SENTENCES:
    print(f"Input: {sentence}")
    entities = extract_gliner(sentence)
    if entities:
        for e in entities:
            print(f"  → {e['text']} ({e['type']}, score={e['score']})")
    else:
        print("  → (no entities)")
    print()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

gliner_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


=== GLiNER Results ===

Input: Metformin is used to treat Type 2 Diabetes
  → Metformin (DRUG, score=0.977)
  → Type 2 Diabetes (DISEASE, score=0.899)

Input: Patient takes Aspirin and Lisinopril daily for hypertension
  → Aspirin (DRUG, score=0.912)
  → Lisinopril (DRUG, score=0.95)
  → hypertension (DISEASE, score=0.923)

Input: 65 year old male with history of hypertension and diabetes mellitus on Metformin 500mg
  → hypertension (SYMPTOM, score=0.569)
  → diabetes mellitus (DISEASE, score=0.841)
  → Metformin 500mg (DRUG, score=0.909)

Input: Ibuprofen may cause gastrointestinal bleeding
  → Ibuprofen (DRUG, score=0.982)
  → gastrointestinal bleeding (SYMPTOM, score=0.783)

Input: The weather is nice today
  → (no entities)

